In [ ]:
from google.colab import drive
drive.mount("/content/drive")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install transformers datasets rouge-score --quiet


  Preparing metadata (setup.py) ... done


In [ ]:
from datasets import load_dataset

# CHANGE THIS PATH to your file
data_path = "/content/drive/MyDrive/biomedical_text_generation/data/unseen/sum_unseen.jsonl"

dataset_dict = load_dataset(
    "json",
    data_files={"test": data_path}
)

test_dataset = dataset_dict["test"]

print("Dataset loaded successfully")
print("Number of examples:", len(test_dataset))
print("\nExample record:")
print(test_dataset[0])


Generating test split: 0 examples [00:00, ? examples/s]

Dataset loaded successfully
Number of examples: 3777

Example record:
{'input': "Previous studies have demonstrated corilagin's inhibitory effects on the growth of various cancer cells. Given the limited research on corilagin's impact on ovarian cancer, a particularly deadly gynecological malignancy, this study aimed to investigate corilagin's influence on A2780 ovarian cancer cell apoptosis and its underlying mechanisms. The goal was to evaluate corilagin's potential as a therapeutic agent for ovarian cancer. The results of the CCK8 assay showed that corilagin inhibited the proliferation of A2780 ovarian cancer cells while exhibiting lower toxicity to normal ovarian surface epithelial cells (IOSE80). We found that corilagin significantly altered the A2780 cell cycle, decreasing the proportion of cells in the Gsub0subGsub1sub and Gsub2subM phases and inducing cell cycle arrest in the S phase. At low concentrations, corilagin induced apoptosis in A2780 cells, accompanied by a decline in

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# 🔁 CHANGE MODEL HERE to compare models
# model_checkpoint = "GanjinZero/biobart-v2-base"
#model_checkpoint = "QizhiPei/biot5-base"
# model_checkpoint = "GanjinZero/biobart-base"
model_checkpoint = "/content/drive/MyDrive/biomedical_text_generation/models/biot5_sum_final"
#model_checkpoint = "/content/drive/MyDrive/biomedical_text_generation/models/biov2bart_sum_final"


tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

model = AutoModelForSeq2SeqLM.from_pretrained(
    model_checkpoint
).to("cuda")

model.eval()

print("Model loaded:", model_checkpoint)


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Model loaded: /content/drive/MyDrive/biomedical_text_generation/models/biot5_sum_final


In [ ]:
def generate_summary(input_text,
                     max_input_length=512,
                     max_new_tokens=256,
                     num_beams=4):

    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        max_length=max_input_length
    ).to("cuda")

    with torch.no_grad():
        summary_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=num_beams,
            early_stopping=True
        )

    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)


In [ ]:
# 🔁 Change this number each time
index = 3775

input_text = test_dataset[index]["input"]
target_text = test_dataset[index]["target"]

generated_text = generate_summary(input_text)

print("\n" + "="*100)
print("MODEL:", model_checkpoint)
print("INDEX:", index)
print("="*100)

print("\nINPUT:\n")
print(input_text)

print("\nTARGET (REFERENCE):\n")
print(target_text)

print("\nGENERATED (MODEL OUTPUT):\n")
print(generated_text)

print("\n" + "="*100)



MODEL: /content/drive/MyDrive/biomedical_text_generation/models/biot5_sum_final
INDEX: 3775

INPUT:

We sought to determine the significance of age and frailty in predicting perioperative outcomes of robotic pancreaticoduodenectomy (RPD). Data from our institution's prospectively collected robotic pancreaticoduodenectomy database was analyzed for the years 20182023. The 5factor modified frailty index (mFI5) was used as a concise stratification tool for frailty. Predictive models for composite adverse event (CAE) variable were created using adjusted logistic regressions. 116 patients underwent RPD. Mean age of this cohort was 70.65 years (11.44). The mean operative time was 311.47 min (71.35) and the estimated blood loss was 107.07 mL (128.49). The most common postoperative complications included in the CAE were pancreatic leak (n  10, 8.62 ), delayed gastric emptying (n  10, 8.62 ), bleeding (n  5, 4.31 ), and atrial fibrillation (n  2, 1.72 ). The 90day mortality was 1.72 . There was

In [ ]:
# 🔁 Change this number each time
index = 3775

input_text = "summarize: " + test_dataset[index]["input"]
target_text = test_dataset[index]["target"]

generated_text = generate_summary(input_text)

print("\n" + "="*100)
print("MODEL:", model_checkpoint)
print("INDEX:", index)
print("="*100)

print("\nINPUT:\n")
print(input_text)

print("\nTARGET (REFERENCE):\n")
print(target_text)

print("\nGENERATED (MODEL OUTPUT):\n")
print(generated_text)

print("\n" + "="*100)



MODEL: /content/drive/MyDrive/biomedical_text_generation/models/biot5_sum_final
INDEX: 3775

INPUT:

summarize: We sought to determine the significance of age and frailty in predicting perioperative outcomes of robotic pancreaticoduodenectomy (RPD). Data from our institution's prospectively collected robotic pancreaticoduodenectomy database was analyzed for the years 20182023. The 5factor modified frailty index (mFI5) was used as a concise stratification tool for frailty. Predictive models for composite adverse event (CAE) variable were created using adjusted logistic regressions. 116 patients underwent RPD. Mean age of this cohort was 70.65 years (11.44). The mean operative time was 311.47 min (71.35) and the estimated blood loss was 107.07 mL (128.49). The most common postoperative complications included in the CAE were pancreatic leak (n  10, 8.62 ), delayed gastric emptying (n  10, 8.62 ), bleeding (n  5, 4.31 ), and atrial fibrillation (n  2, 1.72 ). The 90day mortality was 1.72 

In [ ]:
import json

output_path = "/content/drive/MyDrive/biomedical_text_generation/test_predictions.json"

results = []

for i in range(len(test_dataset)):

    input_text = test_dataset[i]["input"]
    target_text = test_dataset[i]["target"]

    generated_text = generate_summary(input_text)

    results.append({
        "input": input_text,
        "target": target_text,
        "generated": generated_text
    })

with open(output_path, "w") as f:
    json.dump(results, f, indent=2)

print("Results saved to:", output_path)
